In [1]:
"""
HealthPulse Data Merge Script
Combines all dimension tables into one flat file for modeling
"""

import pandas as pd
import numpy as np

print("="*60)
print("HEALTHPULSE DATA MERGE")
print("="*60)

# =============================================================================
# 1. LOAD ALL DATASETS
# =============================================================================

print("\n[1] Loading datasets...")

# Load fact table (main appointments)
fact_appt = pd.read_csv('FACT_APPOINTMENTS.csv')
print(f"  - Fact appointments: {len(fact_appt)} rows")

# Load dimension tables
dim_patients = pd.read_csv('DIM_PATIENTS.csv')
print(f"  - Dim patients: {len(dim_patients)} rows")

dim_providers = pd.read_csv('DIM_PROVIDERS.csv')
print(f"  - Dim providers: {len(dim_providers)} rows")

dim_clinics = pd.read_csv('DIM_CLINICS.csv')  # Note: filename has typo 'clinis'
print(f"  - Dim clinics: {len(dim_clinics)} rows")

dim_dates = pd.read_csv('DIM_DATES.csv')
print(f"  - Dim dates: {len(dim_dates)} rows")

# =============================================================================
# 2. STANDARDIZE COLUMN NAMES
# =============================================================================

print("\n[2] Standardizing column names...")

# Rename columns to match across datasets
# Fact table has CLINIC_ID, dim_clinics has provider_clinic_id
dim_clinics = dim_clinics.rename(columns={
    'provider_clinic_id': 'CLINIC_ID',
    'clinic_name': 'CLINIC_NAME',
    'city': 'CITY',
    'hours': 'HOURS'
})

# Dim providers has CLINIC_ID already, good!
# Dim patients has PATIENT_ID, INSURANCE_TYPE, AGE

# =============================================================================
# 3. MERGE STEP BY STEP
# =============================================================================

print("\n[3] Merging datasets...")

# Start with fact table
merged = fact_appt.copy()
print(f"  - Started with fact table: {len(merged)} rows")

# Merge with patients
merged = merged.merge(
    dim_patients[['PATIENT_ID', 'INSURANCE_TYPE', 'AGE']], 
    on='PATIENT_ID', 
    how='left'
)
print(f"  - After merging patients: {len(merged)} rows")

# Merge with providers (to get specialty)
merged = merged.merge(
    dim_providers[['PROVIDER_ID', 'SPECIALTY']], 
    on='PROVIDER_ID', 
    how='left'
)
print(f"  - After merging providers: {len(merged)} rows")

# Merge with clinics (to get clinic details)
merged = merged.merge(
    dim_clinics[['CLINIC_ID', 'CLINIC_NAME', 'CITY', 'HOURS', 'clinic_assignment']], 
    on='CLINIC_ID', 
    how='left'
)
print(f"  - After merging clinics: {len(merged)} rows")

# =============================================================================
# 4. CREATE FINAL DATASET IN THE FORMAT WE NEED
# =============================================================================

print("\n[4] Creating final dataset format...")

# Map DATE_ID to actual date if needed (simplified - using APPOINTMENT_DATE from fact)
# The dim_dates table would have more date attributes, but we'll use what we have

# Create final dataframe with all columns in our original format
final_df = merged[[
    'APPOINTMENT_ID', 'PATIENT_ID', 'PROVIDER_ID', 
    'APPOINTMENT_DATE', 'APPOINTMENT_TIME', 
    'LEAD_TIME_DAYS', 'WAIT_TIME_MINUTES', 'IS_NO_SHOW',
    'AGE', 'INSURANCE_TYPE', 'SPECIALTY',
    'CLINIC_ID', 'clinic_assignment', 'CLINIC_NAME', 'CITY', 'HOURS'
]].copy()

# Rename columns to match our original working file
final_df = final_df.rename(columns={
    'APPOINTMENT_ID': 'appointment_id',
    'PATIENT_ID': 'patient_id',
    'PROVIDER_ID': 'provider_id',
    'APPOINTMENT_DATE': 'appointment_date',
    'APPOINTMENT_TIME': 'appointment_time',
    'LEAD_TIME_DAYS': 'lead_time_days',
    'WAIT_TIME_MINUTES': 'wait_time_minutes',
    'IS_NO_SHOW': 'is_no_show_0_1',
    'AGE': 'age',
    'INSURANCE_TYPE': 'insurance_type',
    'SPECIALTY': 'specialty',
    'CLINIC_ID': 'provider_clinic_id',
    'CLINIC_ASSIGNMENT': 'clinic_assignment',
    'CLINIC_NAME': 'clinic_name',
    'CITY': 'city',
    'HOURS': 'hours'
})

# Add the metadata columns (these were in original)
final_df['ingestion_timestamp'] = pd.Timestamp.now().timestamp() * 1000
final_df['source_system'] = 'merged_dataset'
final_df['batch_id'] = 'merge_' + pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')

# =============================================================================
# 5. CHECK FOR MISSING VALUES
# =============================================================================

print("\n[5] Checking data quality...")

missing_counts = final_df.isnull().sum()
missing_cols = missing_counts[missing_counts > 0]

if len(missing_cols) > 0:
    print("  ⚠️ Missing values found:")
    for col, count in missing_cols.items():
        print(f"    - {col}: {count} missing ({count/len(final_df)*100:.1f}%)")
else:
    print("  ✅ No missing values!")

# Handle missing values (especially wait_time_minutes)
print("\n[6] Handling missing values...")

# Fill missing wait_time_minutes with median by specialty
if final_df['wait_time_minutes'].isnull().any():
    # Calculate median per specialty
    specialty_medians = final_df.groupby('specialty')['wait_time_minutes'].median()
    
    # Fill missing values
    for specialty, median_wait in specialty_medians.items():
        mask = (final_df['specialty'] == specialty) & (final_df['wait_time_minutes'].isnull())
        final_df.loc[mask, 'wait_time_minutes'] = median_wait
    
    # If still missing, use global median
    global_median = final_df['wait_time_minutes'].median()
    final_df['wait_time_minutes'].fillna(global_median, inplace=True)
    
    print(f"  ✅ Filled missing wait times (used specialty medians)")

# Fill any other missing values
final_df.fillna({
    'insurance_type': 'Unknown',
    'specialty': 'Unknown',
    'clinic_assignment': 'Unknown'
}, inplace=True)

# =============================================================================
# 6. SAVE FINAL DATASET
# =============================================================================

print("\n[7] Saving final dataset...")

output_file = 'healthpulse_db_merged.csv'
final_df.to_csv(output_file, index=False)

print(f"\n✅ MERGE COMPLETE!")
print(f"   Final dataset: {len(final_df)} rows, {len(final_df.columns)} columns")
print(f"   Saved to: {output_file}")

# Show sample
print("\n[Sample of merged data (first 3 rows)]:")
print(final_df[['appointment_id', 'patient_id', 'age', 'insurance_type', 
                'specialty', 'wait_time_minutes', 'is_no_show_0_1']].head(3))

# Show statistics
print("\n[Dataset Statistics]:")
print(f"  - Total appointments: {len(final_df)}")
print(f"  - No-show rate: {final_df['is_no_show_0_1'].mean()*100:.1f}%")
print(f"  - Unique patients: {final_df['patient_id'].nunique()}")
print(f"  - Unique providers: {final_df['provider_id'].nunique()}")
print(f"  - Clinics: {final_df['clinic_name'].nunique()}")
print(f"  - Specialties: {final_df['specialty'].nunique()}")
print(f"  - Avg lead time: {final_df['lead_time_days'].mean():.1f} days")
print(f"  - Avg wait time: {final_df['wait_time_minutes'].mean():.1f} minutes")

print("\n" + "="*60)

HEALTHPULSE DATA MERGE

[1] Loading datasets...
  - Fact appointments: 120000 rows
  - Dim patients: 5000 rows
  - Dim providers: 220 rows
  - Dim clinics: 120000 rows
  - Dim dates: 1096 rows

[2] Standardizing column names...

[3] Merging datasets...
  - Started with fact table: 120000 rows
  - After merging patients: 120000 rows
  - After merging providers: 120000 rows


MemoryError: Unable to allocate 8.84 GiB for an array with shape (1186741264,) and data type int64